# Pangolin environmental raster stack from Google Earth Engine

This notebook prepares environmental predictors for the pangolin RSF workflow.

It uses the new `hsa.remote_sensing` helpers for Earth Engine initialization, spatial diagnostics, temporal diagnostics, and conversion to xarray/Zarr, while keeping project-specific predictor choices visible in the notebook.

Output target:

```text
pango_env_32733.zarr
```

## 0. Setup

Run this from the repository root after installing the package in editable mode:

```bash
conda activate hsa
pip install -e .
```

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from shapely import Polygon

from hsa.remote_sensing import (
    initialize_earth_engine,
    ee_image_to_xarray_stack,
    spatial_summary,
    temporal_summary,
)
from hsa.compute import make_local_dask_client, suggest_xy_chunks, write_raster_stack_zarr


In [ ]:
# Optional local Dask client. Useful for writing/chunking the resulting Zarr stack.
client = make_local_dask_client(n_workers=4, threads_per_worker=1, local_directory='dask-tmp')
client

## 1. Project settings and AOI

In [ ]:
EE_PROJECT = 'test-with-greta'  # change if needed
TARGET_CRS = 'EPSG:32733'
EXPORT_SCALE = 30
BUFFER_M = 10_000

SHAPES = Path('shapefiles')
PERIMETER_FILE = SHAPES / 'OKJ_Okonjima_Nature_Reserve.shp'
OUT_ZARR = Path('pango_env_32733.zarr')

START = '2024-07-01'
END = '2026-04-30'


In [ ]:
ee = initialize_earth_engine(project=EE_PROJECT)
import geemap

perimeter_raw = gpd.read_file(PERIMETER_FILE, engine='pyogrio')
perimeter = gpd.GeoDataFrame(
    geometry=[Polygon(perimeter_raw.geometry.iloc[0])],
    crs=perimeter_raw.crs,
).to_crs(TARGET_CRS)

aoi = gpd.GeoDataFrame(geometry=perimeter.geometry.buffer(BUFFER_M), crs=TARGET_CRS)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
aoi_ee = geemap.geopandas_to_ee(aoi_wgs84)

perimeter.plot(facecolor='none', edgecolor='black')
aoi.boundary.plot(ax=plt.gca(), color='red')
plt.gca().set_aspect('equal')
plt.show()

## 2. Helper functions for Earth Engine images

These helpers are project-specific enough to stay in the notebook for now. Once stable, generic pieces can move into `hsa.remote_sensing`.

In [ ]:
def rename_single_band(img, name):
    return img.rename(name)


def mask_s2_sr(img):
    # Scene Classification Layer masking: cloud shadow, medium/high cloud, cirrus, snow.
    scl = img.select('SCL')
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return img.updateMask(mask).divide(10_000).copyProperties(img, ['system:time_start'])


def add_s2_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('ndvi')
    ndmi = img.normalizedDifference(['B8', 'B11']).rename('ndmi')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('ndwi')
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('mndwi')
    savi = img.expression(
        '1.5 * ((nir - red) / (nir + red + 0.5))',
        {'nir': img.select('B8'), 'red': img.select('B4')},
    ).rename('savi')
    return img.addBands([ndvi, ndmi, ndwi, mndwi, savi])


def local_sd(image, radius_m, name):
    kernel = ee.Kernel.circle(radius=radius_m, units='meters')
    return image.reduceNeighborhood(ee.Reducer.stdDev(), kernel).rename(name)


def local_mean(image, radius_m, name):
    kernel = ee.Kernel.circle(radius=radius_m, units='meters')
    return image.reduceNeighborhood(ee.Reducer.mean(), kernel).rename(name)


def local_cv(image, radius_m, name):
    kernel = ee.Kernel.circle(radius=radius_m, units='meters')
    mean = image.reduceNeighborhood(ee.Reducer.mean(), kernel)
    sd = image.reduceNeighborhood(ee.Reducer.stdDev(), kernel)
    return sd.divide(mean.abs().add(1e-6)).rename(name)


def residual_patchiness(image, broad_radius_m, name):
    broad = local_mean(image, broad_radius_m, name + '_broad')
    return image.subtract(broad).rename(name)

## 3. Sentinel-2 vegetation and moisture predictors

In [ ]:
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
    .map(mask_s2_sr)
    .map(add_s2_indices)
)

s2_indices = s2.select(['ndvi', 'ndmi', 'ndwi', 'mndwi', 'savi'])

veg_mean = s2_indices.mean().rename(['ndvi_mean', 'ndmi_mean', 'ndwi_mean', 'mndwi_mean', 'savi_mean'])
veg_median = s2_indices.median().rename(['ndvi_median', 'ndmi_median', 'ndwi_median', 'mndwi_median', 'savi_median'])
veg_sd = s2_indices.reduce(ee.Reducer.stdDev()).rename(['ndvi_sd_t', 'ndmi_sd_t', 'ndwi_sd_t', 'mndwi_sd_t', 'savi_sd_t'])

vegetation_stack = ee.Image.cat([veg_mean, veg_median, veg_sd])
vegetation_stack.bandNames().getInfo()

## 4. Sentinel-1 texture / structure predictors

In [ ]:
s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)

vv = s1.select('VV').median().rename('vv_median')
vh = s1.select('VH').median().rename('vh_median')
vvvh = vv.subtract(vh).rename('vv_vh_diff')

s1_stack = ee.Image.cat([vv, vh, vvvh])
s1_stack.bandNames().getInfo()

## 5. Land cover, terrain, climate, soil, and water predictors

In [ ]:
# Dynamic World: modal class and class probabilities.
dw = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
)

dw_label = dw.select('label').mode().rename('dw_label')
dw_probs = dw.select([
    'water', 'trees', 'grass', 'flooded_vegetation', 'crops',
    'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
]).mean().rename([
    'dw_water', 'dw_trees', 'dw_grass', 'dw_flooded_vegetation', 'dw_crops',
    'dw_shrub_and_scrub', 'dw_built', 'dw_bare', 'dw_snow_and_ice'
])
lc_stack = ee.Image.cat([dw_label, dw_probs])

# Terrain from SRTM.
dem = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('elevation')
terrain = ee.Terrain.products(dem)
terrain_stack = ee.Image.cat([
    dem,
    terrain.select('slope').rename('slope'),
    terrain.select('aspect').rename('aspect'),
])

# Climate summaries from WorldClim BIO.
bio = ee.Image('WORLDCLIM/V1/BIO')
climate_stack = bio.select(['bio01', 'bio12', 'bio15']).rename([
    'bio01_mean_temp',
    'bio12_annual_precip',
    'bio15_precip_seasonality',
])

# SoilGrids-style global soil predictors. Adjust or replace with iSDA if needed.
# Keep this minimal for a package trial; project-specific soil products can be swapped in later.
soil_ph = ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02').select('b0').rename('soil_ph_h2o')
soil_sand = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_sand')
soil_clay = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_clay')
soil_stack = ee.Image.cat([soil_ph, soil_sand, soil_clay])

# Water occurrence and distance-to-water proxy.
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
water_occurrence = jrc.select('occurrence').rename('water_occurrence')
potential_water = water_occurrence.gt(5).selfMask()
dist2water = potential_water.fastDistanceTransform(1024).sqrt().multiply(ee.Image.pixelArea().sqrt()).rename('dist2water')
water_stack = ee.Image.cat([water_occurrence, dist2water])

base_stack = ee.Image.cat([lc_stack, terrain_stack, climate_stack, soil_stack, water_stack])
base_stack.bandNames().getInfo()

## 6. Spatial patchiness predictors

These are important because pangolin habitat selection may respond not only to absolute vegetation values but also to local heterogeneity.

In [ ]:
ndvi_mean = veg_mean.select('ndvi_mean')
vh_median = s1_stack.select('vh_median')
vvvh_diff = s1_stack.select('vv_vh_diff')

patchiness_stack = ee.Image.cat([
    local_sd(ndvi_mean, 30, 'ndvi_sd_30m'),
    local_sd(ndvi_mean, 90, 'ndvi_sd_90m'),
    local_cv(ndvi_mean, 90, 'ndvi_cv_90m'),
    residual_patchiness(ndvi_mean, 300, 'ndvi_resid_300m'),
    local_sd(vh_median, 30, 'vh_sd_30m'),
    local_sd(vvvh_diff, 30, 'vvvh_sd_30m'),
])

patchiness_stack.bandNames().getInfo()

## 7. Combine predictor stack

At this point, all predictors are still in Earth Engine. The stack is converted to xarray only once near the end.

In [ ]:
predictor_stack = ee.Image.cat([
    vegetation_stack,
    s1_stack,
    base_stack,
    patchiness_stack,
]).clip(aoi_ee.geometry())

bands = predictor_stack.bandNames().getInfo()
len(bands), bands

## 8. Diagnostics: spatial and temporal variability

These cells use the new package helpers. They are diagnostic: they help decide which spatial scales and temporal summaries are biologically useful before fitting the RSF.

In [ ]:
summary_30, sample_30 = spatial_summary(
    ndvi_mean,
    aoi_ee.geometry(),
    band='ndvi_mean',
    scale=30,
    projection=TARGET_CRS,
    target_crs=TARGET_CRS,
    num_pixels=3000,
    include_variogram=True,
)
summary_30

In [ ]:
sample_30.plot(column='ndvi_mean', markersize=3, legend=True, figsize=(7, 6))
perimeter.boundary.plot(ax=plt.gca(), color='black', linewidth=1)
plt.gca().set_aspect('equal')
plt.title('Sampled NDVI values for spatial diagnostic')
plt.show()

In [ ]:
wide_ndvi, long_ndvi = temporal_summary(
    s2_indices.select('ndvi'),
    aoi_ee.geometry(),
    band='ndvi',
    scale=30,
    start=START,
    end=END,
    cadence='MS',
    projection=TARGET_CRS,
    reducers=('mean', 'median', 'stdDev', 'min', 'max'),
)
wide_ndvi.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wide_ndvi['period_mid'], wide_ndvi['mean'], marker='o', label='mean')
ax.plot(wide_ndvi['period_mid'], wide_ndvi['median'], marker='o', label='median')
ax.fill_between(wide_ndvi['period_mid'], wide_ndvi['min'], wide_ndvi['max'], alpha=0.2, label='min-max')
ax.set_ylabel('NDVI')
ax.set_title('Monthly NDVI variability in AOI')
ax.legend()
plt.show()

## 9. Convert Earth Engine image to xarray and write Zarr

This is the bridge into the package RSF notebook. The resulting Zarr file can be opened with `open_raster_stack_zarr()` in `pangolin_rsf_package_trial.ipynb`.

In [ ]:
env = ee_image_to_xarray_stack(
    predictor_stack,
    geometry=aoi_ee.geometry(),
    crs=TARGET_CRS,
    scale=EXPORT_SCALE,
)

env = env.rio.write_crs(TARGET_CRS)
env

In [ ]:
# Quick raster sanity check before writing.
display(env.band.values[:20])
display(env.sizes)

env.sel(band='ndvi_mean').plot(figsize=(7, 6), robust=True)
plt.title('NDVI mean before Zarr export')
plt.show()

In [ ]:
write_raster_stack_zarr(
    env,
    OUT_ZARR,
    name='env',
    target_chunk_mb=256,
    consolidated=True,
)
OUT_ZARR

## 10. Next step

Now open `notebooks/pangolin_rsf_package_trial.ipynb` and use this Zarr file as:

```python
ENV_ZARR = Path('pango_env_32733.zarr')
```

Important: inspect the band names before defining `FeatureSpec`. If a band already carries a scale suffix such as `ndvi_sd_30m`, multiscale sampling may create names like `ndvi_sd_30m_90m`. That is technically unambiguous but may need naming cleanup later.